# Tutorial 05 — Multi-Turn Sessions

**Optional API key** — **Part 6 only** runs three adapter `run_turn` calls on one `session_id` when
`OPENAI_API_KEY` is set. Parts 1–5 are fully local.

A "session" in eXo-brain is more than a single prompt/response pair. This tutorial shows:
- How to build a session-aware adapter that tracks conversation history across turns
- How the `RuntimeTimeline` threads correlation IDs through every event
- How `TenantQuotaManager` enforces per-tenant active-job limits across turns
- What a `QuotaDecision(allowed=False)` looks like when the limit is reached

eXo-brain's built-in `OpenAIAgentsRuntimeAdapter` (PyPI `exo-adapter-openai`, shim at
`src/runtime/openai_agents_runtime.py`) handles session lifecycle for provider-path turns.
For a **local, key-free demo**, Part 2 uses a minimal inline `SessionAdapter` that models
adapter-owned session history (not the Tutorial 02 delegating-wrapper tool path).

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env", override=False)
except ImportError:
    pass

import importlib
import importlib.util

_ADAPTER_WHEELS = (
    ("exo-brain-core-contracts", "exo_brain_core_contracts"),
    ("exo-brain-adapter-sdk", "exo_brain_adapter_sdk"),
    ("exo-adapter-echo", "exo_adapter_echo"),
    ("exo-adapter-openai", "exo_adapter_openai"),
)


def _print_adapter_wheels() -> None:
    for dist, module_name in _ADAPTER_WHEELS:
        if importlib.util.find_spec(module_name) is None:
            print(f"warn: {dist} not installed — pip install -r requirements.txt")
            continue
        mod = importlib.import_module(module_name)
        mod_file = (mod.__file__ or "").replace("\\", "/")
        if "site-packages" not in mod_file and "dist-packages" not in mod_file:
            raise RuntimeError(f"{dist} must be a PyPI wheel in site-packages, got {mod.__file__}")
        if "/eXo_adapters/" in mod_file:
            raise RuntimeError(
                f"{dist} must not load from eXo_adapters checkout — "
                f"pip install -r requirements.txt: {mod.__file__}"
            )
        print(f"{dist}:", mod.__file__)


_print_adapter_wheels()

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

assert OpenAIAgentsRuntimeAdapter.__module__.startswith("exo_adapter_openai."), (
    "OpenAIAgentsRuntimeAdapter must come from exo-adapter-openai (PyPI); "
    "reinstall: pip install -r requirements.txt"
)

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

print("OpenAIAgentsRuntimeAdapter module:", OpenAIAgentsRuntimeAdapter.__module__)
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
HAS_API_KEY = bool(OPENAI_API_KEY)
print(f"API key present: {HAS_API_KEY}")

exo-brain-core-contracts: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_core_contracts/__init__.py
exo-brain-adapter-sdk: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_adapter_sdk/__init__.py
exo-adapter-echo: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_echo/__init__.py
exo-adapter-openai: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_openai/__init__.py
OpenAIAgentsRuntimeAdapter module: exo_adapter_openai.runtime
API key present: True


## Part 1 — Wire the session infrastructure

We wire a `RuntimeTimeline` and `TenantQuotaManager` alongside the adapter.
These are independent of the model provider and work across any adapter.

In [2]:
from src.observability.timeline import RuntimeTimeline
from src.tenancy.quotas import TenantQuotaManager, QuotaDecision
from src.observability.logging import StructuredLogger
from src.observability.metrics import RuntimeMetrics

# Timeline tracks ordered events across all turns of a session
timeline = RuntimeTimeline()
logger = StructuredLogger()
metrics = RuntimeMetrics()

# Quota manager: allow at most 2 concurrent active jobs per tenant
quota_manager = TenantQuotaManager(max_active_jobs_per_tenant=2, hard_enforcement=True)

assert isinstance(timeline, RuntimeTimeline)
assert isinstance(logger, StructuredLogger)
assert isinstance(metrics, RuntimeMetrics)
assert isinstance(quota_manager, TenantQuotaManager)
assert quota_manager.max_active_jobs == 2

print("timeline     :", type(timeline).__name__)
print("quota_manager:", type(quota_manager).__name__, "| max_active_jobs:", quota_manager.max_active_jobs)

timeline     : RuntimeTimeline
quota_manager: TenantQuotaManager | max_active_jobs: 2


## Part 2 — Build a session-aware adapter with history tracking

The adapter layer owns conversation history keyed by `session_id`. A provider adapter can
read that history on later turns; the framework controls what enters the session.

We define a minimal version of the adapter that tracks history without requiring
an API key.

In [3]:
import asyncio
from importlib import import_module

try:
    import_module("nest_asyncio").apply()
except ModuleNotFoundError:
    pass

from typing import Any, AsyncIterator

class SessionAdapter:
    """Minimal session-aware adapter for multi-turn demonstration.
    Tracks conversation history per session without requiring a model provider.
    """

    def __init__(self) -> None:
        self._sessions: dict[str, dict[str, Any]] = {}

    async def start_session(
        self,
        session_id: str,
        tenant_id: str = "default",
        metadata: dict[str, Any] | None = None,
    ) -> None:
        self._sessions[session_id] = {
            "tenant_id": tenant_id,
            "metadata": metadata or {},
            "history": [],  # list of {"role": ..., "content": ...} dicts
        }

    def record_turn(
        self,
        session_id: str,
        user_message: str,
        assistant_reply: str,
    ) -> int:
        """Append a turn to history. Returns new history length."""
        history = self._sessions[session_id]["history"]
        history.append({"role": "user",      "content": user_message})
        history.append({"role": "assistant", "content": assistant_reply})
        return len(history)

    def get_session(self, session_id: str) -> dict[str, Any]:
        return self._sessions[session_id]

    def history_len(self, session_id: str) -> int:
        return len(self._sessions[session_id]["history"])

# Create adapter and start a session
session_adapter = SessionAdapter()
session_id = "session-multiturn-demo"

async def start():
    await session_adapter.start_session(
        session_id=session_id,
        tenant_id="tenant-acme",
        metadata={"purpose": "multi-turn demo"},
    )

asyncio.run(start())

session_data = session_adapter.get_session(session_id)
assert set(session_data.keys()) == {"tenant_id", "metadata", "history"}
assert session_data["tenant_id"] == "tenant-acme"
assert session_data["metadata"] == {"purpose": "multi-turn demo"}
assert session_data["history"] == []

print("Session keys :", list(session_data.keys()))
print("History length (before turns):", session_adapter.history_len(session_id))
print("Tenant ID    :", session_data["tenant_id"])

Session keys : ['tenant_id', 'metadata', 'history']
History length (before turns): 0
Tenant ID    : tenant-acme


## Part 3 — History grows with each turn

Each `record_turn` call appends a user + assistant pair to the session history.
A provider adapter can use this accumulated history as input on later turns,
allowing it to reference previous context.

In [4]:
# Simulate 3 conversation turns
turns = [
    ("What is the capital of France?",   "The capital of France is Paris."),
    ("And what about Germany?",           "The capital of Germany is Berlin."),
    ("Which has more letters in its name?", "Berlin has 6 letters; Paris has 5. Berlin has more."),
]

for i, (user_msg, assistant_reply) in enumerate(turns, 1):
    history_len = session_adapter.record_turn(session_id, user_msg, assistant_reply)
    print(f"Turn {i}: history length = {history_len}")
    print(f"  User     : {user_msg}")
    print(f"  Assistant: {assistant_reply}")
    print()

# Show full history structure
history = session_adapter.get_session(session_id)["history"]
assert len(history) == 6
assert history[0] == {"role": "user", "content": "What is the capital of France?"}
assert history[1] == {"role": "assistant", "content": "The capital of France is Paris."}
assert history[-1]["role"] == "assistant"
assert "Berlin has more" in history[-1]["content"]

print(f"Total history entries: {len(history)}")
print(f"(= {len(history) // 2} turns × 2 messages each)")

Turn 1: history length = 2
  User     : What is the capital of France?
  Assistant: The capital of France is Paris.

Turn 2: history length = 4
  User     : And what about Germany?
  Assistant: The capital of Germany is Berlin.

Turn 3: history length = 6
  User     : Which has more letters in its name?
  Assistant: Berlin has 6 letters; Paris has 5. Berlin has more.

Total history entries: 6
(= 3 turns × 2 messages each)


## Part 4 — Correlation IDs thread through the timeline

Each turn appends events to the `RuntimeTimeline` using a per-turn correlation ID.
`timeline.entries_for(correlation_id)` retrieves all events for that specific turn.
`timeline.all_entries()` gives the complete ordered trace across all turns.

In [5]:
# Record timeline events for each turn (mirrors what a production adapter would do)
for i in range(1, 4):
    corr = f"turn-{session_id}-{i:03d}"
    timeline.append(
        correlation_id=corr,
        event="session.turn_started",
        payload={"session_id": session_id, "turn": i, "tenant_id": "tenant-acme"},
    )
    timeline.append(
        correlation_id=corr,
        event="session.turn_completed",
        payload={"session_id": session_id, "turn": i, "status": "success"},
    )

# Inspect per-turn events
for i in range(1, 4):
    corr = f"turn-{session_id}-{i:03d}"
    entries = timeline.entries_for(corr)
    print(f"Turn {i} ({corr[:30]}...): {len(entries)} events")
    for e in entries:
        print(f"  {e.event}")

print(f"\nTotal timeline entries across all turns: {len(timeline.all_entries())}")

all_entries = timeline.all_entries()
assert len(all_entries) == 6

for i in range(1, 4):
    corr = f"turn-{session_id}-{i:03d}"
    entries = timeline.entries_for(corr)
    assert len(entries) == 2
    assert [e.event for e in entries] == [
        "session.turn_started",
        "session.turn_completed",
    ]
    assert entries[0].payload["session_id"] == session_id
    assert entries[0].payload["turn"] == i
    assert entries[0].payload["tenant_id"] == "tenant-acme"
    assert entries[1].payload["status"] == "success"

print("PASS — correlation IDs thread through the timeline correctly")

Turn 1 (turn-session-multiturn-demo-00...): 2 events
  session.turn_started
  session.turn_completed
Turn 2 (turn-session-multiturn-demo-00...): 2 events
  session.turn_started
  session.turn_completed
Turn 3 (turn-session-multiturn-demo-00...): 2 events
  session.turn_started
  session.turn_completed

Total timeline entries across all turns: 6
PASS — correlation IDs thread through the timeline correctly


## Part 5 — Quota enforcement: allowed and denied

`TenantQuotaManager.check_submission(tenant_id, active_jobs)` enforces the per-tenant
active job limit. It returns a `QuotaDecision` with `allowed`, `reason_code`, and `message`.

This same check runs before each background job submission — making it equally relevant
to multi-turn sessions that submit background work per turn.

In [6]:
TENANT = "tenant-acme"

# Under limit — allowed
decision_ok = quota_manager.check_submission(tenant_id=TENANT, active_jobs=0)
print("active_jobs=0 :", decision_ok)
assert decision_ok.allowed, "Should be allowed when under limit"
assert decision_ok.reason_code == ""
assert decision_ok.message == ""

decision_ok2 = quota_manager.check_submission(tenant_id=TENANT, active_jobs=1)
print("active_jobs=1 :", decision_ok2)
assert decision_ok2.allowed, "Should be allowed at 1 (limit is 2)"
assert decision_ok2.reason_code == ""
assert decision_ok2.message == ""

# At limit — hard enforcement blocks submission
decision_denied = quota_manager.check_submission(tenant_id=TENANT, active_jobs=2)
print("active_jobs=2 :", decision_denied)
assert not decision_denied.allowed, "Should be denied at limit"
assert decision_denied.reason_code == "TENANT_QUOTA_EXCEEDED"
assert "tenant-acme" in decision_denied.message

print()
print("PASS — quota_manager enforces limits correctly")
print(f"Denied reason_code : {decision_denied.reason_code}")
print(f"Denied message     : {decision_denied.message}")

active_jobs=0 : QuotaDecision(allowed=True, reason_code='', message='')
active_jobs=1 : QuotaDecision(allowed=True, reason_code='', message='')
active_jobs=2 : QuotaDecision(allowed=False, reason_code='TENANT_QUOTA_EXCEEDED', message="Tenant 'tenant-acme' exceeded max active jobs quota.")

PASS — quota_manager enforces limits correctly
Denied reason_code : TENANT_QUOTA_EXCEEDED
Denied message     : Tenant 'tenant-acme' exceeded max active jobs quota.


## Part 6 — Optional adapter turns on one session [API key-dependent provider path]

**Already proved without a key (Parts 1–5):**

| Part | What you saw |
|------|----------------|
| 3 | `SessionAdapter` history grows 2 → 4 → 6; simulated Berlin vs Paris comparison |
| 4 | Six timeline entries, one correlation ID per turn |
| 5 | `TENANT_QUOTA_EXCEEDED` at `active_jobs=2` |

**This cell adds:** three `Orchestrator.run_turn` calls on the **same** `session_id` via
`OpenAIAgentsRuntimeAdapter` — expect `output_delta` and `run_complete` each turn, plus a local
history list that grows like Part 3.

**This cell does not prove:**

- **Cross-turn model memory** — each turn currently sends only that turn's user text to the SDK (prior
  messages are not passed in). Turn 3 may ignore turns 1–2; compare to Part 3's scripted answers.
- **Live model responses** — adapter output may be echo/fallback text depending on provider path and key.
- **Tool execution on the orchestrator stream** — no tools are registered here; for governed tool
  proofs use **Tutorial 08** (`planned_tool_call`).

**Skip when `OPENAI_API_KEY` is unset.**

In [7]:
if not HAS_API_KEY:
    print("Skipping live turns — OPENAI_API_KEY not set.")
else:
    from src.core.orchestrator import Orchestrator
    from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
    from src.policies.middleware import DeterministicFirstPolicyMiddleware
    from src.tools.executor import DeterministicToolExecutor
    from src.tools.registry import ToolRegistry
    from src.schemas.events import RuntimeEventType

    registry_live = ToolRegistry()
    policy_live = DeterministicFirstPolicyMiddleware()
    executor_live = DeterministicToolExecutor(registry=registry_live, policy=policy_live)

    live_session_id = "session-live-multiturn-05"
    live_history: list[dict[str, str]] = []

    async def run_live_turns():
        adapter_live = OpenAIAgentsRuntimeAdapter(provider_id="openai-gpt4o-mini")
        health = await adapter_live.healthcheck()
        caps = adapter_live.get_capabilities()
        print("adapter health:", health.state.value, health.reason)
        print("provider_id:", caps.provider_id)

        orch = Orchestrator(
            runtime_adapter=adapter_live,
            policy_middleware=policy_live,
            tool_executor=executor_live,
        )

        session_metadata = {
            "tenant_id": "tenant-acme",
            "agent_id": "nb-live-multiturn",
            "model": "gpt-4o-mini",
            "instructions": "You are a concise assistant. Answer in one short sentence.",
        }

        prompts = [
            "What is the capital of France? One short sentence.",
            "What is the capital of Germany? One short sentence.",
            "Name one European capital with more than five letters. One short sentence.",
        ]

        for i, prompt in enumerate(prompts, 1):
            print(f"\n--- Adapter turn {i} (session={live_session_id}) ---")
            print(f"User: {prompt}")
            live_history.append({"role": "user", "content": prompt})

            reply_parts: list[str] = []
            event_types: list[str] = []
            async for event in orch.run_turn(
                session_id=live_session_id,
                user_input=prompt,
                context={
                    "run_id": f"run-live-{i}",
                    "job_id": "job-live",
                    "task_id": "task-live",
                    "agent_id": "agent-live",
                    "session_metadata": dict(session_metadata),
                },
            ):
                event_types.append(event.event_type.value)
                if event.event_type == RuntimeEventType.OUTPUT_DELTA:
                    text = str(event.payload.get("text", ""))
                    if text:
                        reply_parts.append(text)

            assert RuntimeEventType.OUTPUT_DELTA.value in event_types
            assert RuntimeEventType.RUN_COMPLETE.value in event_types

            reply = "".join(reply_parts) or "(no text delta)"
            live_history.append({"role": "assistant", "content": reply})
            print(f"Assistant: {reply[:160]}")
            print("Events seen:", ", ".join(dict.fromkeys(event_types)))
            print(f"Local history length: {len(live_history)}")

        assert len(live_history) == 6

        print(
            "\nADAPTER SESSION VERIFICATION: PASS — three run_turn calls used one session_id; "
            "local notebook history length is 6."
        )
        print(
            "Compare Part 3 (scripted cross-turn reasoning) vs adapter turn 3 (may not use prior turns). "
            "Governed tool proofs: Tutorial 08."
        )

    asyncio.run(run_live_turns())

adapter health: healthy adapter-initialized
provider_id: openai-gpt4o-mini

--- Adapter turn 1 (session=session-live-multiturn-05) ---
User: What is the capital of France? One short sentence.
Assistant: openai-adapter-echo: What is the capital of France? One short sentence.
Events seen: output_delta, run_complete
Local history length: 2

--- Adapter turn 2 (session=session-live-multiturn-05) ---
User: What is the capital of Germany? One short sentence.
Assistant: openai-adapter-echo: What is the capital of Germany? One short sentence.
Events seen: output_delta, run_complete
Local history length: 4

--- Adapter turn 3 (session=session-live-multiturn-05) ---
User: Name one European capital with more than five letters. One short sentence.
Assistant: openai-adapter-echo: Name one European capital with more than five letters. One short sentence.
Events seen: output_delta, run_complete
Local history length: 6

ADAPTER SESSION VERIFICATION: PASS — three run_turn calls used one session_id; loc

## Summary

| Capability | Module | Key API |
|---|---|---|
| Session lifecycle | `src/runtime/openai_agents_runtime` | `OpenAIAgentsRuntimeAdapter.start_session()` |
| Cross-turn history | Local `SessionAdapter` demo / adapter-managed session state | same `session_id` + adapter-owned history |
| Cross-turn correlation | `src/observability/timeline` | `timeline.append()`, `timeline.entries_for()` |
| Quota enforcement | `src/tenancy/quotas` | `quota_manager.check_submission()` |
| Quota denied | `src/tenancy/quotas` | `QuotaDecision(allowed=False, reason_code="TENANT_QUOTA_EXCEEDED")` |

**Key insight:** Session state (conversation history) lives in the adapter layer (Part 2–3).
The `RuntimeTimeline` links every event back to its session via correlation ID (Part 4).
Quota enforcement is stateless (Part 5). Part 6 optionally shows repeated adapter `run_turn`
calls on one `session_id`; provider-side memory is not proven here (Tutorial 02 covers SDK history wiring).

### Next steps
- **Tutorial 06** — Background workflows: long-running DAG jobs with retries and checkpointing
- **Tutorial 07** — Governance and anomaly detection: detect runaway tenants before they impact others

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Local governance lab (no API key) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Live governance contrasts (optional API key) | `tutorial_09_governed_execution_live.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).